## Model core

1. Portfolio expected return
$$E(R_{p}) = \sum_{i=1}^nw_{i}E(R_{i}),$$
where $w_{i}$ - weight, that equals to the share of the securities in the portfolio.

$E(R_{i})$ - return of the particular stock in the portfolio.

2. Portfolio variance

Portfolio dispersion is a process that determines the degree of risk or volatility associated with an investment portfolio. The basic formula for calculating this variance focuses on the relationship between the so-called return variance and the covariation associated with each of the stocks found in the portfolio, as well as the percentage or part of the portfolio that each stock represents.
$$\sigma_{p}^{2} = \sum_{i}^{}\omega_{i}^{2}\sigma_{i}^{2}+\sum_{i}^{}\sum_{j\neq i}^{}\omega_{i}^{}\omega_{j}^{}\sigma_{i}^{}\sigma_{j}^{}\rho_{ij},$$
where $\omega_{i}$ - stock weights; $\sigma_{i}$ - stock return std; $\rho_{ij}$ - correlation coeff. between two stocks.

3. Sharpe ratio
$$\frac{R_{p} - R_{f}}{\sigma_{p}}$$

4. Efficient frontier

<center>
<img src="https://upload.wikimedia.org/wikipedia/commons/e/e1/Markowitz_frontier.jpg?utm_source=en.wikipedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled" width="400" height="200" alt="Efficient frontier">
</center>

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
while project_root.name != 'python' and project_root.parent != project_root:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Downloading and cleaning up the data

In [ ]:
from data import download_tickers_history

TRADING_DAYS_PER_YEAR = 252

# set the date range for the historic data
start_date = datetime(year=2022, month=1, day=1)
end_date = datetime(year=2026, month=1, day=1)
tickers = ['NVDA', 'AAPL', 'PLTR', 'DKNG', 'CAT', 'INTC', 'AMZN', 'TSLA', 'GOOG', 'MSFT']
history = download_tickers_history(start_date, end_date, tickers)

history.head()

print(history.isnull().sum())


## Optimization
$$\min_{w} \quad \frac{1}{2} w^T \Sigma w \quad \text{(minimizing portfolio variance)},$$
with the following constraints:
1. $w^T \mathbf{1} = 1$ - the sum of all weights equals 100% (all capital is distributed). $w^T$
2. $\mu = \mu_{\text{target}}$ - the portfolio must deliver exactly the return we have fixed.
3. If shorts are prohibited, the limit-inequality is added: $w_i \ge 0 \quad \forall i$ (Long-only).

In [ ]:
from data import get_log_returns

def get_cov_matrix(data: pd.DataFrame) -> pd.DataFrame:
    col_mean = data.apply(lambda x: x - x.mean())
    multiplication = col_mean.T.dot(col_mean)

    return multiplication / (col_mean.shape[0])

log_returns = get_log_returns(history)
cov = get_cov_matrix(log_returns)
cov * TRADING_DAYS_PER_YEAR

In [ ]:
from data import get_log_returns

N = history.shape[1]
log_returns = get_log_returns(history)
yr_cov = log_returns.cov() * TRADING_DAYS_PER_YEAR  # type: ignore
weights = np.ones(N) / N;
mu = log_returns.mean() * TRADING_DAYS_PER_YEAR
